<a href="https://colab.research.google.com/github/ayyanarh1/tamil-nadu-school-flood-risk/blob/main/day17_streamlit_570.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Setup
!pip install streamlit streamlit-folium folium pandas numpy plotly -q

import pandas as pd
import numpy as np

# Load the 570 school dataset
# Upload tamil_nadu_570_schools_risk.csv to Colab first
from google.colab import files
print('Please upload tamil_nadu_570_schools_risk.csv')
uploaded = files.upload()

import io
df_570 = pd.read_csv(io.BytesIO(list(uploaded.values())[0]))
print(f'✅ Loaded {len(df_570)} schools!')
print(df_570.head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.2/529.2 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 75.1 MB/s eta 0:00:00
Please upload tamil_nadu_570_schools_risk.csv


Saving tamil_nadu_570_schools_risk.csv to tamil_nadu_570_schools_risk.csv
✅ Loaded 570 schools!
   school_id                     school_name district   connectivity  \
0  IND_00001      Secondary School Chennai 1  Chennai  not_connected   
1  IND_00002        Primary School Chennai 2  Chennai  not_connected   
2  IND_00003        Primary School Chennai 3  Chennai  not_connected   
3  IND_00004        Primary School Chennai 4  Chennai      connected   
4  IND_00005  Upper Primary School Chennai 5  Chennai      connected   

   water_occurrence_pct  flood_frequency  jrc_norm  gfd_norm  flood_score  \
0                  0.00             0.00       0.0       0.0          0.0   
1                 11.73             0.24      11.7       2.0          7.8   
2                 99.24             0.00      99.2       0.0         59.5   
3                  0.00             0.00       0.0       0.0          0.0   
4                 99.33             0.00      99.3       0.0         59.6   

   conn_

In [3]:
# Write updated app with 570 schools
import pandas as pd

# Convert df_570 to dict for embedding in app
schools_dict = df_570.to_dict(orient='records')

app_code = '''
import streamlit as st
import pandas as pd
import folium
from streamlit_folium import st_folium
from folium.plugins import MarkerCluster
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

st.set_page_config(
    page_title="School Flood Risk Dashboard",
    page_icon="🏫",
    layout="wide"
)

st.markdown("""
<style>
    .main-header {
        background: linear-gradient(90deg, #021C3B, #065A82);
        padding: 20px;
        border-radius: 10px;
        color: white;
        margin-bottom: 20px;
    }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="main-header">
    <h1>🏫 School Flood Risk Dashboard</h1>
    <p>UNICEF Giga Initiative — Tamil Nadu + Mozambique Climate Hazard Assessment</p>
    <p>570 Tamil Nadu Schools + 10 Mozambique Schools | 6 Data Sources</p>
</div>
""", unsafe_allow_html=True)

@st.cache_data
def load_data():
    # Tamil Nadu 570 schools
    tn_df = pd.read_csv(
        'https://raw.githubusercontent.com/ayyanarh1/'
        'tamil-nadu-school-flood-risk/main/'
        'tamil_nadu_570_schools_risk.csv'
    )
    tn_df['country'] = 'Tamil Nadu'
    tn_df = tn_df.rename(columns={
        'connectivity': 'connectivity_status'
    })

    # Mozambique 10 schools
    moz_data = {
        'school_name': [
            'School Inhambane Coast',
            'Primary School Quelimane',
            'Secondary School Beira Coast',
            'Primary School Sofala',
            'Primary School Tete',
            'Primary School Gaza',
            'Secondary School Zambezia',
            'Secondary School Nampula',
            'School Cabo Delgado',
            'Primary School Maputo Central'
        ],
        'district': [
            'Inhambane', 'Zambezia', 'Sofala', 'Sofala',
            'Tete', 'Gaza', 'Zambezia', 'Nampula',
            'Cabo Delgado', 'Maputo'
        ],
        'latitude': [
            -23.86, -17.88, -19.84, -19.52, -16.16,
            -24.05, -17.05, -15.12, -12.37, -25.96
        ],
        'longitude': [
            35.38, 36.89, 34.84, 34.56, 33.59,
            34.40, 36.98, 39.27, 40.52, 32.57
        ],
        'connectivity_status': [
            'not_connected', 'not_connected', 'not_connected',
            'not_connected', 'not_connected', 'connected',
            'not_connected', 'connected', 'not_connected',
            'connected'
        ],
        'flood_score': [
            93.2, 56.4, 38.9, 34.9, 54.9,
            54.2, 0.5, 0.0, 0.0, 0.5
        ],
        'final_score': [
            100.2, 86.2, 78.3, 73.3, 61.3,
            53.8, 47.3, 34.7, 25.7, 8.3
        ],
        'risk_tier': [
            'CRITICAL', 'CRITICAL', 'CRITICAL', 'CRITICAL', 'CRITICAL',
            'CRITICAL', 'HIGH', 'HIGH', 'MEDIUM', 'LOW'
        ],
        'school_type': ['Secondary'] * 10,
        'country': ['Mozambique'] * 10
    }
    moz_df = pd.DataFrame(moz_data)

    # Combine
    combined = pd.concat([tn_df, moz_df], ignore_index=True)
    combined['connectivity_status'] = combined[
        'connectivity_status'
    ].fillna('unknown')
    return combined

df = load_data()

# ── Sidebar ──
st.sidebar.header("🔧 Filters")

country = st.sidebar.multiselect(
    "Country",
    options=df["country"].unique(),
    default=df["country"].unique()
)

district = st.sidebar.multiselect(
    "District",
    options=sorted(df["district"].unique()),
    default=sorted(df["district"].unique())
)

risk_filter = st.sidebar.multiselect(
    "Risk tier",
    options=["CRITICAL", "HIGH", "MEDIUM", "LOW"],
    default=["CRITICAL", "HIGH", "MEDIUM", "LOW"]
)

connectivity = st.sidebar.multiselect(
    "Connectivity",
    options=df["connectivity_status"].unique(),
    default=df["connectivity_status"].unique()
)

st.sidebar.markdown("---")
st.sidebar.markdown(
    "[View on GitHub](https://github.com/ayyanarh1/"
    "tamil-nadu-school-flood-risk)"
)
st.sidebar.markdown(
    "[Live App](https://tamil-nadu-school-flood-risk-"
    "ewc2sj7fhrvvtkwlpw5jzf.streamlit.app/)"
)

# Filter
filtered = df[
    (df["country"].isin(country)) &
    (df["district"].isin(district)) &
    (df["risk_tier"].isin(risk_filter)) &
    (df["connectivity_status"].isin(connectivity))
].copy()

# ── Metrics ──
st.subheader("Summary Statistics")
c1, c2, c3, c4, c5 = st.columns(5)
c1.metric("Total schools", len(filtered))
c2.metric("CRITICAL",
    len(filtered[filtered.risk_tier == "CRITICAL"]))
c3.metric("HIGH",
    len(filtered[filtered.risk_tier == "HIGH"]))
c4.metric("No connectivity",
    len(filtered[filtered.connectivity_status == "not_connected"]))
c5.metric("Avg score",
    round(filtered.final_score.mean(), 1)
    if len(filtered) > 0 else 0)

st.markdown("---")

# ── Tabs ──
tab1, tab2, tab3, tab4 = st.tabs([
    "🗺️ Map", "📊 Charts",
    "📋 Data Table", "🏫 School Profile"
])

with tab1:
    st.subheader("Risk Map")

    if len(filtered) > 0:
        center_lat = filtered.latitude.mean()
        center_lon = filtered.longitude.mean()
    else:
        center_lat, center_lon = 10.5, 78.5

    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=6,
        tiles="CartoDB positron"
    )

    color_map = {
        "CRITICAL": "#CC0000",
        "HIGH":     "#FF6600",
        "MEDIUM":   "#FFAA00",
        "LOW":      "#2D8A4E"
    }

    # Use clustering for large datasets
    if len(filtered) > 50:
        cluster = MarkerCluster().add_to(m)
        add_to = cluster
    else:
        add_to = m

    for _, row in filtered.iterrows():
        color = color_map.get(row["risk_tier"], "gray")
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=7,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.8,
            tooltip=row["school_name"],
            popup=folium.Popup(
                f"<b>{row['school_name']}</b><br>"
                f"District: {row['district']}<br>"
                f"Country: {row['country']}<br>"
                f"Connectivity: {row['connectivity_status']}<br>"
                f"Score: {row['final_score']}<br>"
                f"Risk: <b>{row['risk_tier']}</b>",
                max_width=250
            )
        ).add_to(add_to)

    st_folium(m, width=1000, height=500)

with tab2:
    st.subheader("Risk Analysis Charts")

    ch1, ch2 = st.columns(2)

    with ch1:
        # Risk distribution
        risk_counts = filtered["risk_tier"].value_counts().reset_index()
        risk_counts.columns = ["Risk Tier", "Count"]
        fig1 = px.bar(
            risk_counts,
            x="Risk Tier", y="Count",
            color="Risk Tier",
            color_discrete_map=color_map,
            title="Risk Distribution",
            category_orders={"Risk Tier": [
                "CRITICAL", "HIGH", "MEDIUM", "LOW"
            ]}
        )
        fig1.update_layout(showlegend=False, height=350)
        st.plotly_chart(fig1, use_container_width=True)

    with ch2:
        # District risk heatmap
        dist_risk = filtered.groupby(
            ["district", "risk_tier"]
        ).size().reset_index(name="count")
        fig2 = px.bar(
            dist_risk,
            x="district", y="count",
            color="risk_tier",
            color_discrete_map=color_map,
            title="Risk by District",
            category_orders={"risk_tier": [
                "CRITICAL", "HIGH", "MEDIUM", "LOW"
            ]}
        )
        fig2.update_layout(
            height=350,
            xaxis_tickangle=-45
        )
        st.plotly_chart(fig2, use_container_width=True)

    ch3, ch4 = st.columns(2)

    with ch3:
        # Connectivity pie chart
        conn_counts = filtered[
            "connectivity_status"
        ].value_counts().reset_index()
        conn_counts.columns = ["Status", "Count"]
        fig3 = px.pie(
            conn_counts,
            values="Count",
            names="Status",
            title="Connectivity Status",
            color_discrete_sequence=["#2D8A4E", "#CC0000", "#FFAA00"]
        )
        fig3.update_layout(height=300)
        st.plotly_chart(fig3, use_container_width=True)

    with ch4:
        # Score distribution
        fig4 = px.histogram(
            filtered,
            x="final_score",
            color="risk_tier",
            color_discrete_map=color_map,
            title="Risk Score Distribution",
            nbins=20
        )
        fig4.update_layout(height=300)
        st.plotly_chart(fig4, use_container_width=True)

with tab3:
    st.subheader("School Data Table")

    # Top 50 most at risk
    st.markdown(f"Showing top 50 of {len(filtered)} schools by risk score")
    display_df = filtered[[
        "school_name", "district", "country",
        "risk_tier", "final_score",
        "connectivity_status"
    ]].sort_values(
        "final_score", ascending=False
    ).head(50)
    display_df.columns = [
        "School", "District", "Country",
        "Risk", "Score", "Connectivity"
    ]
    st.dataframe(display_df, height=400, hide_index=True)

    # Download
    csv = filtered.to_csv(index=False)
    st.download_button(
        label="Download all filtered schools (CSV)",
        data=csv,
        file_name="filtered_schools.csv",
        mime="text/csv"
    )

with tab4:
    st.subheader("School Profile")
    selected = st.selectbox(
        "Select a school",
        options=filtered["school_name"].tolist()
    )

    if selected:
        school = filtered[
            filtered.school_name == selected
        ].iloc[0]

        st.markdown(f"### {school['school_name']}")

        i1, i2, i3, i4 = st.columns(4)
        i1.metric("District", school["district"])
        i2.metric("Country", school["country"])
        i3.metric("Risk Tier", school["risk_tier"])
        i4.metric("Score", school["final_score"])

        i5, i6 = st.columns(2)
        i5.metric("Connectivity", school["connectivity_status"])
        i6.metric("Flood Score", school["flood_score"])

        # Mini map
        mini_map = folium.Map(
            location=[school["latitude"], school["longitude"]],
            zoom_start=12,
            tiles="CartoDB positron"
        )
        folium.CircleMarker(
            location=[school["latitude"], school["longitude"]],
            radius=15,
            color=color_map.get(school["risk_tier"], "gray"),
            fill=True,
            fill_opacity=0.8,
            tooltip=school["school_name"]
        ).add_to(mini_map)
        st_folium(mini_map, width=600, height=300)

st.markdown("---")
with st.expander("📖 Methodology"):
    st.markdown("""
    ### Risk Framework — IPCC H x E x V
    - **Hazard:** JRC Surface Water + GFD + Sentinel-1 SAR
    - **Exposure:** Coastal district classification
    - **Vulnerability:** Connectivity status

    ### Risk Tiers
    - 🚨 CRITICAL: Score >= 50
    - 🔴 HIGH: Score 25-50
    - 🟡 MEDIUM: Score 10-25
    - 🟢 LOW: Score < 10

    **Full code:** github.com/ayyanarh1/tamil-nadu-school-flood-risk
    """)

st.caption(
    "Data: Sentinel-1, JRC, GFD, ERA5, IBTrACS | "
    "UNICEF Giga Initiative | "
    "github.com/ayyanarh1/tamil-nadu-school-flood-risk"
)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print('✅ Updated app written!')
print('New features:')
print('  - 570 Tamil Nadu + 10 Mozambique schools')
print('  - District filter')
print('  - 4 tabs: Map, Charts, Table, School Profile')
print('  - 4 Plotly charts')
print('  - Mini school map in profile tab')
print('  - Clustering for large datasets')

✅ Updated app written!
New features:
  - 570 Tamil Nadu + 10 Mozambique schools
  - District filter
  - 4 tabs: Map, Charts, Table, School Profile
  - 4 Plotly charts
  - Mini school map in profile tab
  - Clustering for large datasets


In [4]:
# Download files
from google.colab import files

# Updated requirements
with open('requirements.txt', 'w') as f:
    f.write('''streamlit
streamlit-folium
folium
pandas
numpy
plotly
''')

files.download('app.py')
files.download('requirements.txt')
print('✅ Files downloaded!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Files downloaded!
